In [8]:
# your imports go here. Feel free to import other modules.
import sqlite3
import pandas as pd

In [9]:
# connect to database
con = sqlite3.connect("Data/co2_and_population.db")
cur = con.cursor()

In [10]:
# some sample queries for the database
res = cur.execute("SELECT * FROM un_population_estimate LIMIT 5")
for row in res:
    print(row)

(1, 'Total, all countries or areas', 2010, 'Population mid-year estimates (millions)', 7021.73, None, 'United Nations Population Division, New York, World Population Prospects: The 2024 Revision, last accessed July 2024.')
(1, 'Total, all countries or areas', 2010, 'Population mid-year estimates for males (millions)', 3533.14, None, 'United Nations Population Division, New York, World Population Prospects: The 2024 Revision, last accessed July 2024.')
(1, 'Total, all countries or areas', 2010, 'Population mid-year estimates for females (millions)', 3488.59, None, 'United Nations Population Division, New York, World Population Prospects: The 2024 Revision, last accessed July 2024.')
(1, 'Total, all countries or areas', 2010, 'Sex ratio (males per 100 females)', 101.3, None, 'United Nations Population Division, New York, World Population Prospects: The 2024 Revision; supplemented by data from the United Nations Statistics Division, New York, Demographic Yearbook 2022 and Secretariat for 

In [11]:
res = cur.execute("PRAGMA table_info(un_population_estimate)")
res.fetchall()

[(0, 'code', 'INTEGER', 0, None, 0),
 (1, 'region/country/area', 'TEXT', 0, None, 0),
 (2, 'year', 'INTEGER', 0, None, 0),
 (3, 'series', 'TEXT', 0, None, 0),
 (4, 'value', 'REAL', 0, None, 0),
 (5, 'footnotes', 'TEXT', 0, None, 0),
 (6, 'source', 'TEXT', 0, None, 0)]

## Part 1: Importing Data and Calculating UN Emissions per Capita

In [12]:
# ── Load CSVs ─────────────────────────────────────────────────────────────
owid = pd.read_csv("Data/co-emissions-per-capita.csv")
owid.columns = ["entity", "code", "year", "co2_per_capita_t"]

un_raw = pd.read_csv("Data/un_co2_emissions_estimate.csv", skiprows=1, header=0)
un_raw.columns = ["country_code", "country", "year", "series", "value", "footnotes", "source"]
un_raw["value"] = un_raw["value"].astype(str).str.replace(",", "", regex=False).astype(float)

# ── Keep only emissions rows (not per capita) ─────────────────────────────
un_emissions = un_raw[
    un_raw["series"] == "Emissions (thousand metric tons of carbon dioxide)"
][["country_code", "country", "year", "value"]].rename(columns={"value": "emissions_kt"})

# ── Create tables ─────────────────────────────────────────────────────────
cur.executescript("""
    DROP TABLE IF EXISTS co2_per_capita;
    CREATE TABLE co2_per_capita (
        entity TEXT, code TEXT, year INTEGER, co2_per_capita_t REAL
    );
    DROP TABLE IF EXISTS un_co2_emissions;
    CREATE TABLE un_co2_emissions (
        country_code TEXT, country TEXT, year INTEGER, emissions_kt REAL
    );
""")

owid.to_sql("co2_per_capita", con, if_exists="append", index=False)
un_emissions.to_sql("un_co2_emissions", con, if_exists="append", index=False)
con.commit()

# ── Part 1: UN emissions per capita 2010 ──────────────────────────────────
un_2010 = pd.read_sql("""
    SELECT
        e.country,
        e.emissions_kt,
        p.value AS population_millions,
        ROUND((e.emissions_kt * 1000.0) / (p.value * 1000000.0), 4) AS un_t_per_capita
    FROM un_co2_emissions e
    JOIN un_population_estimate p
        ON e.country_code = p.code AND e.year = p.year
    WHERE e.year = 2010
      AND p.series = 'Population mid-year estimates (millions)'
      AND CAST(e.country_code AS INTEGER) > 0
      AND e.country != 'Total, all countries or areas'
      AND e.emissions_kt IS NOT NULL
      AND p.value IS NOT NULL
      AND p.value > 0
    ORDER BY un_t_per_capita DESC
""", con)

print("── UN emissions per capita 2010 (top 20) ──")


un_2010[["country", "un_t_per_capita"]] \
    .rename(columns={"un_t_per_capita": "un_emissions_per_capita"}) \
    .to_csv("un_co2_data.csv", index=False)
print("Saved un_co2_data.csv")

── UN emissions per capita 2010 (top 20) ──
Saved un_co2_data.csv


## Part 2: Data Integration and Comparison to Our World in Data

In [13]:
from rapidfuzz import process, fuzz

# ── Part 2: Fuzzy match to OWID and compare ───────────────────────────────
owid_2010 = owid[owid["year"] == 2010][["entity", "code", "co2_per_capita_t"]].dropna(subset=["code"])
owid_names = owid_2010["entity"].tolist()

# Manual corrections for known bad fuzzy matches
manual_corrections = {
    "Bolivia (Plurin. State of)":       "Bolivia",
    "Dem. People's Rep. Korea":         "North Korea",
    "Republic of Korea":                "South Korea",
    "Iran (Islamic Republic of)":       "Iran",
    "Syrian Arab Republic":             "Syria",
    "Republic of Moldova":              "Moldova",
    "Viet Nam":                         "Vietnam",
    "Russian Federation":               "Russia",
    "United States of America":         "United States",
    "United Rep. of Tanzania":          "Tanzania",
    "Venezuela (Boliv. Rep. of)":       "Venezuela",
    "Lao People's Dem. Rep.":           "Laos",
    "Brunei Darussalam":                "Brunei",
    "Cabo Verde":                       "Cape Verde",
    "Türkiye":                          "Turkey",
    "Côte d'Ivoire":                    "Cote d'Ivoire",
    "Netherlands (Kingdom of the)":     "Netherlands",
    "Micronesia (Fed. States of)":      "Micronesia (country)",
    "State of Palestine":               "Palestine",
    "Cayman Islands":                   "Cayman Islands",
    "American Samoa":                   "American Samoa",
    "Guadeloupe":                       "Guadeloupe",
    "Isle of Man":                      "Isle of Man",
    "Northern Mariana Islands":         "Northern Mariana Islands",
    "Puerto Rico":                      "Puerto Rico",
    "Réunion":                          "Reunion",
    "China, Hong Kong SAR":             "Hong Kong",
    "China, Macao SAR":                 "Macao",
    "Falkland Islands (Malvinas)":      "Falkland Islands",
    "Martinique":                       "Martinique",
    "Mayotte":                          "Mayotte",
    "French Guiana":                    "French Guiana",
}

# Regions to exclude entirely
regions_to_exclude = {
    "Africa", "Asia", "Europe", "South America", "North America",
    "Total, all countries or areas", "Oceania"
}

def get_match(name):
    if name in regions_to_exclude:
        return None
    if name in manual_corrections:
        return manual_corrections[name]
    match, score, _ = process.extractOne(name, owid_names, scorer=fuzz.token_sort_ratio)
    return match if score >= 80 else None

un_2010["owid_match"] = un_2010["country"].apply(get_match)

# Drop unmatched rows
un_2010_filtered = un_2010[un_2010["owid_match"].notna()]


file2 = un_2010_filtered.merge(owid_2010, left_on="owid_match", right_on="entity")
file2 = file2[["code", "un_t_per_capita", "co2_per_capita_t"]].copy()
file2.columns = ["iso3", "un_emissions_per_capita", "owd_emissions_per_capita"]
file2["iso3"] = file2["iso3"].str.upper()
file2 = file2.sort_values("iso3").reset_index(drop=True)

file2.to_csv("integrated_co2_data.csv", index=False)
print("Saved integrated_co2_data.csv")
print(file2.head(10))
print(f"Rows: {len(file2)}")


con.close()

Saved integrated_co2_data.csv
  iso3  un_emissions_per_capita  owd_emissions_per_capita
0  ABW                  40.8800                 25.029972
1  AFG                   0.2985                  0.295742
2  AGO                   0.6413                  0.986628
3  AIA                  13.2000                  9.864194
4  ALB                   1.3898                  1.633426
5  AND                   6.6750                  6.399722
6  ARE                  24.3112                 26.640587
7  ARG                   4.2741                  4.503556
8  ARM                   1.3846                  1.450914
9  ATG                   6.2000                  6.440348
Rows: 204


### Writeup

2-3 sentences of markdown text describing how you integrated the data in step 2. Your paragraph below:

To integrate the two datasets, I joined the UN CO2 emissions data with the UN population estimates using the shared numeric country code and year, which allowed me to calculate emissions per capita in metric tons per person for each country in 2010. I then matched the resulting UN country list to the Our World in Data dataset using fuzzy string matching (via the `rapidfuzz` library), supplemented by a manual corrections dictionary to handle cases where country names differed significantly between sources (e.g. "Viet Nam" → "Vietnam", "Republic of Korea" → "South Korea"). This merge allowed me to retrieve the ISO3 country code from the OWID dataset and align both emissions per capita estimates side by side for comparison.